# 04.0 SimCLR pretraining

## Notebook overview
This notebook pools the defect-free training images across categories and trains the SimCLR encoder checkpoints used later by the downstream SSL anomaly detectors.

## Purpose
- load the split manifest and leakage report created in notebook 01 so the self-supervised stage uses the same clean data partition
- train a self-supervised ResNet-18 encoder on pooled `train_good` images only
- compare the mild and strong augmentation settings under a matched SimCLR setup
- save encoder checkpoints, loss curves, training tables, and JSON summaries for later downstream reuse

## Process
1. load the cleaned train-good split written by notebook 01
2. pool normal training images across the active categories
3. define the SimCLR dataset, augmentations, encoder, projector, and NT-Xent loss
4. train each requested augmentation setting and save the best encoder checkpoint
5. export summary tables and figures so notebook 05 can reuse the saved encoders

## Notes
- this notebook expects `01_dataset_audit_and_splits.ipynb` to have been run first
- the downstream SSL notebook should use the encoder checkpoints saved here rather than retraining them

In [ ]:
#------------------------------------------------------------------------------
# 1.0 Imports and notebook helpers
#------------------------------------------------------------------------------
import os
import sys
import json
import time
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

from tqdm.auto import tqdm

from IPython.display import display


# Print a clean section banner so notebook output is easier to scan.
def print_banner(text):
    print("\n" + "=" * 90)
    print(text)
    print("=" * 90)


# Create a parent folder before saving an artefact.
def ensure_parent(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


# Save JSON with overwrite behaviour.
def save_json_overwrite(obj, path):
    path = ensure_parent(path)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


# Save a DataFrame to CSV with overwrite behaviour.
def save_csv_overwrite(df, path):
    path = ensure_parent(path)
    df.to_csv(path, index=False)


# Read JSON from disk.
def read_json(path):
    with open(Path(path), "r") as f:
        return json.load(f)


# Return a file size in MB.
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


# Set Python, NumPy, and PyTorch seeds.
def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# Reset peak GPU memory before timing a stage.
def reset_peak_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


# Return peak GPU memory in MB if CUDA is available.
def get_peak_gpu_memory_mb():
    if not torch.cuda.is_available():
        return np.nan
    return torch.cuda.max_memory_allocated() / (1024 ** 2)


print_banner("Runtime environment")
print("Python      :", sys.version.split()[0])
print("Torch       :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("CUDA        :", torch.cuda.is_available())
print("GPU count   :", torch.cuda.device_count())
for gpu_idx in range(torch.cuda.device_count()):
    print(f"GPU {gpu_idx}      :", torch.cuda.get_device_name(gpu_idx))

# 2.0 Run settings

## Purpose
- define the active runtime, categories, and SimCLR hyperparameters in one place
- keep notebook outputs organised into portable folders for tables, figures, JSON files, and checkpoints
- make later notebooks easier to reproduce by saving the exact settings used for the SSL stage

In [ ]:
#------------------------------------------------------------------------------
# 2.1 Core configuration and output paths
#------------------------------------------------------------------------------
NOTEBOOK_ID = "04_simclr_pretraining"
RUN_MODE = "full"                  # "debug" or "full"
SEED = 42
DETERMINISTIC_DEBUG = False
set_seed(SEED, deterministic=(RUN_MODE == "debug" and DETERMINISTIC_DEBUG))

DEBUG_CATEGORIES = ["bottle", "carpet", "tile", "transistor"]
CATEGORIES_ALL = [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper",
]
CATEGORIES_ACTIVE = DEBUG_CATEGORIES if RUN_MODE == "debug" else CATEGORIES_ALL

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()

IMG_SIZE = 224
CPU_COUNT = os.cpu_count() or 2
NUM_WORKERS_BASE = min(4, max(2, CPU_COUNT // 2))
NUM_WORKERS = NUM_WORKERS_BASE if DEVICE == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

SIMCLR_BATCH_SIZE = 128 if DEVICE == "cuda" else 32
SIMCLR_EPOCHS = 10 if RUN_MODE == "debug" else 30
SIMCLR_LR = 3e-4
SIMCLR_WEIGHT_DECAY = 1e-4
SIMCLR_TEMPERATURE = 0.2
SIMCLR_PROJ_DIM = 128
SIMCLR_AUG_STRENGTHS = ["strong"] if RUN_MODE == "debug" else ["mild", "strong"]
SIMCLR_SAVE_FULL_MODEL_BACKUP = False
SIMCLR_ENABLE_DP_FALLBACK = False

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

if Path("/kaggle/working").exists():
    WORK_ROOT = Path("/kaggle/working/industrial_anomaly_detection_mvtec")
else:
    WORK_ROOT = Path.cwd() / "industrial_anomaly_detection_mvtec"

NOTEBOOK_ROOT = WORK_ROOT / NOTEBOOK_ID / RUN_MODE
FIGURES_DIR = NOTEBOOK_ROOT / "figures"
TABLES_DIR = NOTEBOOK_ROOT / "tables"
JSON_DIR = NOTEBOOK_ROOT / "json"
CHECKPOINTS_DIR = NOTEBOOK_ROOT / "checkpoints"

for folder in [FIGURES_DIR, TABLES_DIR, JSON_DIR, CHECKPOINTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_01_ROOT = WORK_ROOT / "01_dataset_audit_and_splits" / RUN_MODE
SPLIT_MANIFEST_PATH = NOTEBOOK_01_ROOT / "json" / "split_manifest.json"
LEAKAGE_REPORT_PATH = NOTEBOOK_01_ROOT / "json" / "leakage_report.json"

RUN_METADATA_PATH = JSON_DIR / "run_metadata.json"
SIMCLR_SETTINGS_PATH = JSON_DIR / "simclr_settings.json"
TABLE_SSL_COVERAGE_PATH = TABLES_DIR / "table_ssl_coverage.csv"
TABLE_SIMCLR_SUMMARY_PATH = TABLES_DIR / "table_simclr_summary.csv"
SIMCLR_RUNS_JSON_PATH = JSON_DIR / "simclr_runs.json"

RUN_METADATA = {
    "notebook_id": NOTEBOOK_ID,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": DEVICE,
    "gpu_count": GPU_COUNT,
    "timestamp": datetime.now().isoformat(),
}
save_json_overwrite(RUN_METADATA, RUN_METADATA_PATH)

SIMCLR_SETTINGS = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "categories_active": CATEGORIES_ACTIVE,
    "image_size": IMG_SIZE,
    "batch_size": SIMCLR_BATCH_SIZE,
    "epochs": SIMCLR_EPOCHS,
    "learning_rate": SIMCLR_LR,
    "weight_decay": SIMCLR_WEIGHT_DECAY,
    "temperature": SIMCLR_TEMPERATURE,
    "projection_dim": SIMCLR_PROJ_DIM,
    "augmentation_settings": SIMCLR_AUG_STRENGTHS,
    "save_full_model_backup": SIMCLR_SAVE_FULL_MODEL_BACKUP,
    "data_parallel_fallback": SIMCLR_ENABLE_DP_FALLBACK,
}
save_json_overwrite(SIMCLR_SETTINGS, SIMCLR_SETTINGS_PATH)

print_banner("Core settings")
print("NOTEBOOK_ROOT:", NOTEBOOK_ROOT)
print("RUN_MODE     :", RUN_MODE)
print("DEVICE       :", DEVICE)
print("GPU_COUNT    :", GPU_COUNT)
print("CATEGORIES   :", len(CATEGORIES_ACTIVE))
print("AUG SETTINGS :", SIMCLR_AUG_STRENGTHS)

# 3.0 Dataset and prior artefacts

## Purpose
- resolve the dataset path in a way that works in Kaggle or a local clone
- load the split manifest and leakage report created in notebook 01
- fail clearly if the foundation notebook has not been run first

In [ ]:
#------------------------------------------------------------------------------
# 3.1 Resolve the dataset path and load prior notebook artefacts
#------------------------------------------------------------------------------
DATASET_CANDIDATES = [
    Path(os.environ.get("MVTEC_DIR", "")) if os.environ.get("MVTEC_DIR") else None,
    Path("/kaggle/input/datasets/ipythonx/mvtec-ad"),
    Path("/kaggle/input/mvtec-ad"),
    Path.cwd() / "data" / "raw" / "mvtec-ad",
]

MVTEC_DIR = None
for candidate in DATASET_CANDIDATES:
    if candidate is not None and candidate.exists():
        MVTEC_DIR = candidate
        break

if MVTEC_DIR is None:
    raise FileNotFoundError(
        "MVTec AD dataset path was not found. Set MVTEC_DIR as an environment variable "
        "or attach the dataset in Kaggle before running the notebook."
    )

if not SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Split manifest was not found at {SPLIT_MANIFEST_PATH}. "
        "Run 01_dataset_audit_and_splits.ipynb first."
    )

if not LEAKAGE_REPORT_PATH.exists():
    raise FileNotFoundError(
        f"Leakage report was not found at {LEAKAGE_REPORT_PATH}. "
        "Run 01_dataset_audit_and_splits.ipynb first."
    )

split_manifest = read_json(SPLIT_MANIFEST_PATH)
leakage_report = read_json(LEAKAGE_REPORT_PATH)

if not leakage_report.get("all_checks_zero", False):
    raise AssertionError(
        "Notebook 01 reported non-zero leakage checks. Fix that before running the SimCLR notebook."
    )

missing_categories = sorted(set(CATEGORIES_ACTIVE) - set(split_manifest.keys()))
if len(missing_categories) > 0:
    raise ValueError(
        f"The split manifest does not contain the required categories for this run: {missing_categories}"
    )

available_categories = sorted([path.name for path in MVTEC_DIR.iterdir() if path.is_dir()])

print_banner("Dataset and dependency check")
print("MVTEC_DIR               :", MVTEC_DIR)
print("Available categories    :", len(available_categories))
print("Split manifest path     :", SPLIT_MANIFEST_PATH)
print("Leakage report path     :", LEAKAGE_REPORT_PATH)
print("Leakage checks all zero :", leakage_report["all_checks_zero"])

# 4.0 Shared SimCLR datasets, transforms, and model helpers

## Purpose
- pool the normal training images across categories for the SSL stage
- define the mild and strong augmentation policies used in the study
- build the encoder, projection head, and NT-Xent loss used for pretraining

In [ ]:
#------------------------------------------------------------------------------
# 4.1 Build SSL coverage table, paired-view dataset, augmentations, and model
#------------------------------------------------------------------------------
ssl_rows = []
ssl_paths = []

for category in CATEGORIES_ACTIVE:
    category_paths = split_manifest[category]["train_good"]
    ssl_paths.extend(category_paths)
    ssl_rows.append(
        {
            "category": category,
            "n_train_good": len(category_paths),
        }
    )

df_ssl_coverage = pd.DataFrame(ssl_rows).sort_values("category").reset_index(drop=True)
save_csv_overwrite(df_ssl_coverage, TABLE_SSL_COVERAGE_PATH)

print_banner("SSL training coverage")
display(df_ssl_coverage)
print("Total pooled SSL images:", len(ssl_paths))
print("Saved coverage table   :", TABLE_SSL_COVERAGE_PATH)


# Apply Gaussian blur as one of the SimCLR augmentations.
class GaussianBlur:
    def __init__(self, p=0.5, radius_min=0.1, radius_max=2.0):
        self.p = p
        self.radius_min = radius_min
        self.radius_max = radius_max

    def __call__(self, img):
        if random.random() > self.p:
            return img
        radius = float(random.uniform(self.radius_min, self.radius_max))
        return img.filter(ImageFilter.GaussianBlur(radius=radius))


# Build the mild or strong SimCLR augmentation pipeline.
def build_simclr_transform(img_size, strength="mild"):
    if strength == "mild":
        color_jitter = transforms.ColorJitter(0.20, 0.20, 0.10, 0.05)
        blur = GaussianBlur(p=0.20, radius_min=0.1, radius_max=1.0)
        crop_scale = (0.60, 1.00)
        gray_p = 0.05
        flip_p = 0.50
    elif strength == "strong":
        color_jitter = transforms.ColorJitter(0.40, 0.40, 0.20, 0.10)
        blur = GaussianBlur(p=0.50, radius_min=0.1, radius_max=2.0)
        crop_scale = (0.40, 1.00)
        gray_p = 0.10
        flip_p = 0.50
    else:
        raise ValueError(f"Unknown augmentation strength: {strength}")

    return transforms.Compose(
        [
            transforms.RandomResizedCrop(img_size, scale=crop_scale),
            transforms.RandomHorizontalFlip(p=flip_p),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=gray_p),
            blur,
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )


# Return two augmented views from the same normal image.
class SimclrDataset(Dataset):
    def __init__(self, img_paths, transform):
        self.img_paths = list(img_paths)
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        image = Image.open(self.img_paths[idx]).convert("RGB")
        return self.transform(image), self.transform(image)


# Build the SimCLR DataLoader.
def make_simclr_loader(img_paths, aug_strength):
    dataset = SimclrDataset(
        img_paths=img_paths,
        transform=build_simclr_transform(IMG_SIZE, strength=aug_strength),
    )

    loader_kwargs = dict(
        batch_size=SIMCLR_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        drop_last=True,
    )

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return DataLoader(dataset, **loader_kwargs)


# Build the ResNet-18 encoder used before the projection head.
def get_resnet18_encoder():
    encoder = torchvision.models.resnet18(weights=None)
    feature_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()
    return encoder, feature_dim


# Map encoder features into the contrastive projection space.
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


# Combine the encoder and projector into one SimCLR model.
class SimclrModel(nn.Module):
    def __init__(self, encoder, feat_dim, proj_dim=128):
        super().__init__()
        self.encoder = encoder
        self.projector = ProjectionHead(feat_dim, hidden_dim=512, out_dim=proj_dim)

    def forward(self, x):
        features = self.encoder(x)
        projections = self.projector(features)
        return F.normalize(projections, dim=1)


# Compute the standard SimCLR NT-Xent loss.
def nt_xent_loss(z1, z2, temperature=0.2):
    batch_size = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)

    similarity = torch.matmul(z, z.t()) / temperature
    diagonal = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
    similarity = similarity.masked_fill(diagonal, torch.finfo(similarity.dtype).min)

    targets = torch.arange(2 * batch_size, device=z.device)
    targets = (targets + batch_size) % (2 * batch_size)

    return F.cross_entropy(similarity.float(), targets)

# 5.0 SimCLR training helpers

## Purpose
- train one augmentation setting at a time
- save the best encoder checkpoint based on training loss
- export per-epoch loss tables, summary JSON files, and loss curves for later notebooks

In [ ]:
#------------------------------------------------------------------------------
# 5.1 Training helpers for one SimCLR run
#------------------------------------------------------------------------------
def plot_and_save_loss(df_train, fig_path, aug_strength):
    fig = plt.figure(figsize=(7, 4))
    plt.plot(df_train["epoch"], df_train["loss"], marker="o")
    plt.title(f"SimCLR Training Loss ({aug_strength.capitalize()} Augmentation)")
    plt.xlabel("Epoch")
    plt.ylabel("NT-Xent Loss")
    plt.grid(True)
    plt.tight_layout()
    ensure_parent(fig_path)
    plt.savefig(fig_path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def run_one_simclr_train(aug_strength):
    print_banner(f"SimCLR run: {aug_strength}")

    train_loader = make_simclr_loader(ssl_paths, aug_strength=aug_strength)
    encoder, feat_dim = get_resnet18_encoder()
    model = SimclrModel(encoder, feat_dim, proj_dim=SIMCLR_PROJ_DIM).to(DEVICE)

    if SIMCLR_ENABLE_DP_FALLBACK and DEVICE == "cuda" and GPU_COUNT > 1:
        model = nn.DataParallel(model)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SIMCLR_LR,
        weight_decay=SIMCLR_WEIGHT_DECAY,
    )

    use_amp = DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    encoder_path = CHECKPOINTS_DIR / f"simclr_{aug_strength}_encoder.pt"
    full_model_path = CHECKPOINTS_DIR / f"simclr_{aug_strength}_full.pt"
    loss_table_path = TABLES_DIR / f"table_simclr_train_{aug_strength}.csv"
    figure_path = FIGURES_DIR / f"fig_simclr_loss_{aug_strength}.png"
    meta_path = JSON_DIR / f"simclr_{aug_strength}_meta.json"

    rows = []
    best_loss = np.inf
    best_epoch = None

    reset_peak_gpu_memory()
    fit_start = time.time()

    for epoch in range(1, SIMCLR_EPOCHS + 1):
        model.train()
        epoch_start = time.time()
        batch_losses = []

        progress = tqdm(
            train_loader,
            desc=f"{aug_strength} | epoch {epoch:02d}/{SIMCLR_EPOCHS}",
            leave=False,
        )

        for x1, x2 in progress:
            x1 = x1.to(DEVICE, non_blocking=(DEVICE == "cuda"))
            x2 = x2.to(DEVICE, non_blocking=(DEVICE == "cuda"))

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp,
            ):
                x = torch.cat([x1, x2], dim=0)
                z = model(x)
                z1, z2 = torch.chunk(z, 2, dim=0)
                loss = nt_xent_loss(z1, z2, temperature=SIMCLR_TEMPERATURE)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            batch_losses.append(loss.item())
            progress.set_postfix(loss=float(np.mean(batch_losses)))

        epoch_loss = float(np.mean(batch_losses)) if batch_losses else np.nan
        epoch_sec = float(time.time() - epoch_start)

        rows.append(
            {
                "epoch": epoch,
                "loss": epoch_loss,
                "sec": epoch_sec,
                "aug_strength": aug_strength,
                "run_mode": RUN_MODE,
            }
        )

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_epoch = epoch

            save_model = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(save_model.encoder.state_dict(), encoder_path)

            if SIMCLR_SAVE_FULL_MODEL_BACKUP:
                torch.save(save_model.state_dict(), full_model_path)

        print(f"Epoch {epoch:02d} | loss={epoch_loss:.4f} | sec={epoch_sec:.1f}")

    total_fit_sec = float(time.time() - fit_start)
    peak_gpu_mem_mb = get_peak_gpu_memory_mb()

    df_train = pd.DataFrame(rows)
    save_csv_overwrite(df_train, loss_table_path)
    plot_and_save_loss(df_train, figure_path, aug_strength=aug_strength)

    meta = {
        "run_mode": RUN_MODE,
        "seed": SEED,
        "aug_strength": aug_strength,
        "categories_ssl": CATEGORIES_ACTIVE,
        "n_categories_ssl": len(CATEGORIES_ACTIVE),
        "n_train_images_ssl": len(ssl_paths),
        "image_size": IMG_SIZE,
        "batch_size": SIMCLR_BATCH_SIZE,
        "epochs": SIMCLR_EPOCHS,
        "learning_rate": SIMCLR_LR,
        "weight_decay": SIMCLR_WEIGHT_DECAY,
        "temperature": SIMCLR_TEMPERATURE,
        "projection_dim": SIMCLR_PROJ_DIM,
        "backbone_name": "resnet18",
        "amp_enabled": use_amp,
        "data_parallel_used": isinstance(model, nn.DataParallel),
        "total_train_sec": total_fit_sec,
        "mean_epoch_sec": float(df_train["sec"].mean()) if len(df_train) > 0 else np.nan,
        "best_loss": float(best_loss),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "peak_gpu_mem_mb": float(peak_gpu_mem_mb) if not np.isnan(peak_gpu_mem_mb) else None,
        "encoder_checkpoint_path": str(encoder_path),
        "full_model_checkpoint_path": str(full_model_path) if SIMCLR_SAVE_FULL_MODEL_BACKUP else None,
        "loss_table_path": str(loss_table_path),
        "loss_figure_path": str(figure_path),
        "checkpoint_size_mb": file_size_mb(encoder_path),
    }
    save_json_overwrite(meta, meta_path)

    return {
        "method": f"simclr_{aug_strength}",
        "aug_strength": aug_strength,
        "run_mode": RUN_MODE,
        "n_categories_ssl": len(CATEGORIES_ACTIVE),
        "n_train_images_ssl": len(ssl_paths),
        "epochs": SIMCLR_EPOCHS,
        "batch_size": SIMCLR_BATCH_SIZE,
        "learning_rate": SIMCLR_LR,
        "temperature": SIMCLR_TEMPERATURE,
        "best_loss": float(best_loss),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "total_train_sec": total_fit_sec,
        "mean_epoch_sec": float(df_train["sec"].mean()) if len(df_train) > 0 else np.nan,
        "checkpoint_size_mb": file_size_mb(encoder_path),
        "peak_gpu_mem_mb": peak_gpu_mem_mb,
        "encoder_checkpoint_path": str(encoder_path),
        "loss_table_path": str(loss_table_path),
        "loss_figure_path": str(figure_path),
        "meta_path": str(meta_path),
    }

# 6.0 Run SimCLR pretraining

## Purpose
- train the requested augmentation settings in sequence
- save a concise summary table that later notebooks can read without re-parsing multiple JSON files

In [ ]:
#------------------------------------------------------------------------------
# 6.1 Train the requested SimCLR settings and save summary artefacts
#------------------------------------------------------------------------------
if len(ssl_paths) < SIMCLR_BATCH_SIZE and RUN_MODE == "full":
    raise ValueError(
        "The pooled SSL dataset is smaller than the requested batch size. "
        "Reduce SIMCLR_BATCH_SIZE before training."
    )

summary_rows = []
for aug_strength in SIMCLR_AUG_STRENGTHS:
    summary_rows.append(run_one_simclr_train(aug_strength))

df_simclr_summary = pd.DataFrame(summary_rows).sort_values("aug_strength").reset_index(drop=True)
save_csv_overwrite(df_simclr_summary, TABLE_SIMCLR_SUMMARY_PATH)

ssl_ckpt_map = {
    aug_strength: str(CHECKPOINTS_DIR / f"simclr_{aug_strength}_encoder.pt")
    for aug_strength in SIMCLR_AUG_STRENGTHS
}
available_ssl_tags = [
    aug_strength
    for aug_strength in SIMCLR_AUG_STRENGTHS
    if (CHECKPOINTS_DIR / f"simclr_{aug_strength}_encoder.pt").exists()
]

runs_payload = {
    "timestamp": datetime.now().isoformat(),
    "run_mode": RUN_MODE,
    "categories_ssl": CATEGORIES_ACTIVE,
    "n_train_images_ssl": len(ssl_paths),
    "available_ssl_tags": available_ssl_tags,
    "ssl_checkpoint_map": ssl_ckpt_map,
    "runs": df_simclr_summary.to_dict(orient="records"),
}
save_json_overwrite(runs_payload, SIMCLR_RUNS_JSON_PATH)

print_banner("SimCLR summary")
display(df_simclr_summary)
print("Saved summary table :", TABLE_SIMCLR_SUMMARY_PATH)
print("Saved runs JSON     :", SIMCLR_RUNS_JSON_PATH)
print("Available checkpoints:", available_ssl_tags)

# 7.0 Final review

## Purpose
- list the most important artefacts written by this notebook so the downstream SSL notebook can reuse them cleanly

In [ ]:
#------------------------------------------------------------------------------
# 7.1 Final saved artefacts
#------------------------------------------------------------------------------
print_banner("Saved artefacts")
print("Run metadata path        :", RUN_METADATA_PATH)
print("SimCLR settings path     :", SIMCLR_SETTINGS_PATH)
print("Split manifest dependency:", SPLIT_MANIFEST_PATH)
print("Leakage report dependency:", LEAKAGE_REPORT_PATH)
print("SSL coverage table path  :", TABLE_SSL_COVERAGE_PATH)
print("SimCLR summary table path:", TABLE_SIMCLR_SUMMARY_PATH)
print("SimCLR runs JSON path    :", SIMCLR_RUNS_JSON_PATH)
print("Checkpoint folder        :", CHECKPOINTS_DIR)
print("Figures folder           :", FIGURES_DIR)
print("Tables folder            :", TABLES_DIR)
print("JSON folder              :", JSON_DIR)